# Demo 5 — PoT vs CoT bake-off on financial reasoning
## Session 6: Advanced Prompt Engineering — bonus demo

> **MLflow as the tape recorder.** Every prompt sent, every response returned, every generated Python program, every subprocess stdout, every token count — captured automatically by `mlflow.openai.autolog()` and the `@mlflow.trace` decorators below. Run the cells, then open the MLflow UI to replay every input and output the model saw.

**The setup:** LLMs are unreliable arithmeticians. Interpreters are not.

Watch a Chain-of-Thought prompt confidently *reason* through a multi-step financial calculation — laying out every step, sounding completely competent — and then **botch the final answer by a decimal place**. Then watch Program-of-Thought (LLM writes Python, we exec it in a sandbox) nail the same problem in 4 lines.

If you have a numeric workload anywhere in your pipeline — token cost projections, p95 latency, FX-adjusted revenue, anything with more than two arithmetic ops — **this is the cheapest, highest-ROI swap you can make**. One prompt change. ~1.2× the tokens. Goodbye decimal-place bugs.

**Citation:** Chen et al. 2022, *Program of Thoughts Prompting* — [arXiv:2211.12588](https://arxiv.org/abs/2211.12588) (TMLR 2023). Headline numbers:
- **GSM8K: PoT 71.6% vs CoT 63.1% (+8.5pp)**
- **FinQA: PoT 64.5% vs CoT 40.4% (+24.1pp)** ← the killer number for any engineer touching tabular numeric data

**Cell composition:** ~$0.03 of `gpt-4o-mini` for the full live run.

## Setup — env-var driven config

Everything below is configured from environment variables. Set `OPENAI_API_KEY` (or `OPENAI_API_KEY`) for live calls; leave it unset for mock mode. Override `MLFLOW_TRACKING_URI`, `MLFLOW_EXPERIMENT`, `OPENAI_BASE_URL`, `MODEL` to point at any OpenAI-compatible endpoint (OpenAI, Azure, vLLM, Ollama, Together, etc.) and any MLflow server.

```bash
export MLFLOW_TRACKING_URI=http://127.0.0.1:5000
export OPENAI_API_KEY=sk-...
export MODEL=gpt-4o-mini
```

**Local-by-default.** Notebook ships with `OPENAI_BASE_URL=http://localhost:11434/v1` and `MODEL=qwen2.5-coder:7b` (Ollama — PoT works on cheap models since the interpreter does the math). Set the env vars to point at OpenAI, Anthropic, vLLM, Together, etc. — any OpenAI-compatible endpoint.

**Cost tracking uses hypothetical local-LLM rates** (~$0.00005/1K tokens) so the cost-aware traces and scorers still produce meaningful numbers in the demos. Override the `PRICING` dict with your real per-1K rates for production.

In [ ]:
import os
import re
import subprocess
import tempfile
import textwrap
import time
from typing import Optional

import pandas as pd
import mlflow
from mlflow.entities import SpanType
from openai import OpenAI

# --- MLflow tracking ---------------------------------------------------------
MLFLOW_TRACKING_URI      = os.getenv("MLFLOW_TRACKING_URI", "http://127.0.0.1:5000")
MLFLOW_TRACKING_USERNAME = os.getenv("MLFLOW_TRACKING_USERNAME")  # optional (Databricks/remote)
MLFLOW_TRACKING_PASSWORD = os.getenv("MLFLOW_TRACKING_PASSWORD")  # optional
MLFLOW_REGISTRY_URI      = os.getenv("MLFLOW_REGISTRY_URI", MLFLOW_TRACKING_URI)
MLFLOW_EXPERIMENT        = os.getenv("MLFLOW_EXPERIMENT", "session6/demo_05_pot_vs_cot")

if MLFLOW_TRACKING_USERNAME:
    os.environ["MLFLOW_TRACKING_USERNAME"] = MLFLOW_TRACKING_USERNAME
if MLFLOW_TRACKING_PASSWORD:
    os.environ["MLFLOW_TRACKING_PASSWORD"] = MLFLOW_TRACKING_PASSWORD

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_registry_uri(MLFLOW_REGISTRY_URI)
mlflow.set_experiment(MLFLOW_EXPERIMENT)

# --- LLM (OpenAI-compatible: OpenAI, Azure, vLLM, Ollama, Together, etc.) ---
OPENAI_BASE_URL = os.getenv("OPENAI_BASE_URL", "http://localhost:11434/v1")  # Ollama default
OPENAI_API_KEY  = os.getenv("OPENAI_API_KEY", "ollama")  # Ollama ignores; any non-empty string works
MODEL    = os.getenv("MODEL", "qwen2.5-coder:7b")  # PoT works on cheap models — interpreter does math

# Reproducibility default. Both CoT and PoT use this. PoT's executor MUST stay at 0.0.
TEMPERATURE = float(os.getenv("TEMPERATURE", "0.0"))

# Per-1K-token pricing (USD). Update at each model release.
PRICING = {
    "gpt-4o":             {"in": 0.0025,   "out": 0.010},
    "gpt-4o-mini":        {"in": 0.00015,  "out": 0.0006},
    "o1-mini":            {"in": 0.003,    "out": 0.012},
    "claude-sonnet-4-6":  {"in": 0.003,    "out": 0.015},
    # Hypothetical local-LLM pricing (electricity + amortised GPU per 1K tokens — adjust to your infra)
    "qwen2.5-coder:14b":  {"in": 0.00005,  "out": 0.00005},
    "qwen2.5-coder:7b":   {"in": 0.00002,  "out": 0.00002},
    "llama3.1:8b":        {"in": 0.00002,  "out": 0.00002},
}

def price_of(model: str, usage) -> float:
    """USD cost for a single chat call. 0.0 in mock mode (no usage).
    Unknown models fall back to hypothetical local-LLM rates (~$0.00005/1K)."""
    if usage is None:
        return 0.0
    p = PRICING.get(model, {"in": 0.00005, "out": 0.00005})  # hypothetical fallback
    return usage.prompt_tokens / 1000 * p["in"] + usage.completion_tokens / 1000 * p["out"]


def tag_cost_latency(latency_ms: float, cost_usd: float) -> None:
    """Attach cost + latency to the active span and root trace."""
    span = mlflow.get_current_active_span()
    if span:
        span.set_attribute("latency_ms", latency_ms)
        span.set_attribute("cost_usd", cost_usd)
    try:
        mlflow.update_current_trace(tags={
            "cost_usd":   f"{cost_usd:.6f}",
            "latency_ms": f"{latency_ms:.0f}",
        })
    except Exception:
        pass


client = OpenAI(base_url=OPENAI_BASE_URL, api_key=OPENAI_API_KEY or "no-key")

# --- Turn MLflow into the tape recorder -------------------------------------
# Captures every chat.completions.create call — request, response, tokens, latency, cost.
mlflow.openai.autolog()

USE_MOCK = not OPENAI_API_KEY
print(f"MLflow:  {MLFLOW_TRACKING_URI}  (exp: {MLFLOW_EXPERIMENT})")
print(f"LLM:     {OPENAI_BASE_URL}  model={MODEL}  mock={USE_MOCK}  temperature={TEMPERATURE}")
print(f"UI:      open {MLFLOW_TRACKING_URI}  -> Experiments -> {MLFLOW_EXPERIMENT}")
print(f"NOTE: defaults to local Ollama (http://localhost:11434/v1). Set OPENAI_BASE_URL/OPENAI_API_KEY/MODEL to point at OpenAI/Anthropic/vLLM.")

if USE_MOCK:
    print("\nWARNING: no OPENAI_API_KEY / OPENAI_API_KEY set — running in MOCK mode.")
    print("         CoT answers are pre-canned (some wrong); PoT code is pre-canned (correct).")

## The eval set — 6 realistic engineering-flavoured financial questions

Not contrived word problems. These are workloads you actually see in code reviews and capacity planning meetings:

| # | Task | Why CoT struggles |
|---|------|-------------------|
| 1 | p95 latency from 20 raw measurements | Linear interpolation between two indices — easy to round mid-calc |
| 2 | Quarterly revenue across 3 segments × 3 compounding steps | 9 multiplications, easy to slip a digit |
| 3 | Annual LLM token bill with 15% buffer | Big numbers × small unit price × 365 — decimal place hell |
| 4 | YoY FX-adjusted gross margin in percentage points | Cross-currency + division + difference of two ratios |
| 5 | Cloud bill: 4 instance types + storage + egress + 18% reserved discount | Lots of multi-line addition; discount applied to wrong subtotal is the classic bug |
| 6 | Compound interest, varying rates, 5 years | Five chained multiplications. One rounding error compounds. |

Gold answers were computed in Python *first* — the LLM is being scored against ground truth, not against the demo author's mental arithmetic.

In [ ]:
# -- The eval set with gold answers (computed in Python beforehand) --------

eval_data = [
    {
        "id": "q1_p95_latency",
        "question": (
            "Given these 20 API request latencies in milliseconds: "
            "[42, 51, 38, 67, 49, 55, 71, 44, 58, 62, 39, 48, 53, 61, 47, 89, 52, 56, 41, 73], "
            "compute the p95 latency using linear interpolation between the two nearest "
            "ranks (numpy's default 'linear' method). Give the answer in milliseconds."
        ),
        "answer": 73.8,
    },
    {
        "id": "q2_quarterly_revenue",
        "question": (
            "A SaaS business has three revenue segments. Starting from Q1: "
            "Segment A is $120,000 and grows 8% quarter-over-quarter; "
            "Segment B is $85,000 and grows 12% QoQ; "
            "Segment C is $45,000 and grows 5% QoQ. "
            "What is the total revenue (sum of all three segments) in Q4 "
            "(i.e. after three quarters of compounding growth from Q1)? "
            "Give the answer in dollars, rounded to 2 decimal places."
        ),
        "answer": 322677.45,
    },
    {
        "id": "q3_token_cost",
        "question": (
            "Project the annual LLM API bill: 1,200,000 input tokens per day, "
            "priced at $2.50 per 1,000,000 tokens, over 365 days, "
            "with a 15% buffer multiplier applied to the final total. "
            "Give the answer in dollars, rounded to 2 decimal places."
        ),
        "answer": 1259.25,
    },
    {
        "id": "q4_fx_gross_margin",
        "question": (
            "Compute the year-over-year change in gross margin percentage (in percentage points) "
            "for an EU subsidiary, reported in USD. "
            "This year: revenue EUR 4,250,000, COGS EUR 1,890,000, FX rate 1 EUR = 1.08 USD. "
            "Last year: revenue EUR 3,800,000, COGS EUR 1,710,000, FX rate 1 EUR = 1.12 USD. "
            "Gross margin % = (revenue - COGS) / revenue * 100. "
            "Report (this year's GM%) - (last year's GM%), in percentage points, rounded to 4 decimals."
        ),
        "answer": 0.5294,
    },
    {
        "id": "q5_cloud_bill",
        "question": (
            "Compute the monthly cloud bill. Compute fleet (730 hours/month each): "
            "12 x m5.large at $0.096/hr, 6 x m5.xlarge at $0.192/hr, "
            "4 x c5.2xlarge at $0.340/hr, 8 x r5.large at $0.126/hr. "
            "Apply an 18% reserved-instance discount to the compute subtotal ONLY. "
            "Then add storage: 15 TB of EBS gp3 at $0.08 per GB-month (1 TB = 1024 GB). "
            "Then add egress: 8 TB at $0.09 per GB. "
            "Give the final monthly bill in dollars, rounded to 2 decimals."
        ),
        "answer": 4762.74,
    },
    {
        "id": "q6_compound_interest",
        "question": (
            "A principal of $250,000 is invested with annual compounding and full reinvestment. "
            "The annual rates over five years are: "
            "Year 1: 4.2%, Year 2: 4.8%, Year 3: 5.1%, Year 4: 4.5%, Year 5: 5.3%. "
            "What is the final balance after five years? Round to 2 decimals."
        ),
        "answer": 315730.39,
    },
]

print(f"Eval set: {len(eval_data)} questions with hard-coded gold answers.")
for q in eval_data:
    print(f"  {q['id']:25s}  gold = {q['answer']}")

## CoT baseline — confident reasoning, wrong answer

Zero-shot Chain-of-Thought. We ask the model to *think step by step* and put the numeric answer on the last line. We then parse the last number.

**What to watch for:** the model will lay out the reasoning fluently — *"first I compute the compute subtotal as $X, then apply the 18% discount, then add storage…"* — and somewhere in those nine multiplications it will round to the wrong decimal, miscount zeros, or apply the discount to the wrong subtotal. The reasoning *looks* right. The answer is off.

Scoring rule: within **±0.5% of gold** = correct. Generous tolerance — we're not punishing legitimate rounding.

In [ ]:
# -- CoT: "Let's think step by step" ---------------------------------------

COT_PROMPT = (
    "Solve the following problem. Let's think step by step. "
    "Show your reasoning, then state the final numeric answer (just the number, "
    "no units, no commas, no dollar signs) on the LAST line by itself.\n\n"
    "Problem: {question}"
)

_NUM_RE = re.compile(r"-?\d+(?:\.\d+)?(?:[eE][+-]?\d+)?")


def parse_last_number(text: str) -> Optional[float]:
    """Pull the last number out of the response. Tolerates trailing punctuation/units."""
    for line in reversed(text.strip().splitlines()):
        cleaned = line.replace(",", "").replace("$", "").replace("%", "").strip()
        m = list(_NUM_RE.finditer(cleaned))
        if m:
            try:
                return float(m[-1].group())
            except ValueError:
                continue
    return None


# Mock CoT answers — pre-canned to show the failure mode.
# Some are correct; the interesting ones are off by a decimal place / rounding compounding.
_MOCK_COT = {
    "q1_p95_latency":         ("Sort the list, take index 0.95*(20-1)=18.05. "
                                "Between s[18]=73 and s[19]=89, interpolate: 73 + 0.05*(89-73) = 73 + 0.8 = 73.8. "
                                "Wait, 0.05 * 16 = 0.8, so answer is 73.8. "
                                "Actually let me recompute: 73 + 0.05 * 16 = 73 + 8 = 81.\n81",  # decimal-place botch
                                 320),
    "q2_quarterly_revenue":   ("Q4 A = 120000 * 1.08^3 = 120000 * 1.2597 = 151164.96. "
                                "Q4 B = 85000 * 1.12^3 = 85000 * 1.4049 = 119417.28. "
                                "Q4 C = 45000 * 1.05^3 = 45000 * 1.1576 = 52092.56. "
                                "Total = 151164.96 + 119417.28 + 52092.56 = 322674.80.\n322674.80",  # close but slight rounding error
                                 240),
    "q3_token_cost":          ("Daily cost = 1.2M * $2.50 / 1M = $3.00. "
                                "Annual = $3.00 * 365 = $1095. "
                                "With 15% buffer = $1095 * 1.15 = $1259.25. "
                                "Hmm, but 1.2M tokens at $2.50/M is $0.30, not $3.00. Let me redo. "
                                "Daily = $0.30 * 365 = $109.50 * 1.15 = $125.93.\n125.93",  # off by 10x — classic decimal slip
                                 280),
    "q4_fx_gross_margin":     ("This year: rev USD = 4250000*1.08 = 4590000, COGS USD = 1890000*1.08 = 2041200. "
                                "GM% = (4590000-2041200)/4590000 * 100 = 2548800/4590000 * 100 = 55.53. "
                                "Last year: rev USD = 3800000*1.12 = 4256000, COGS USD = 1710000*1.12 = 1915200. "
                                "GM% = (4256000-1915200)/4256000 * 100 = 2340800/4256000 * 100 = 55.00. "
                                "Delta = 55.53 - 55.00 = 0.53 pp.\n0.53",  # right magnitude, wrong precision (gold = 0.5294)
                                 310),
    "q5_cloud_bill":          ("Compute: 12*730*0.096 = 840.96. 6*730*0.192 = 840.96. "
                                "4*730*0.34 = 992.80. 8*730*0.126 = 735.84. "
                                "Subtotal = 840.96+840.96+992.80+735.84 = 3410.56. "
                                "After 18% discount: 3410.56 * 0.82 = 2796.66. "
                                "Storage: 15*1024*0.08 = 1228.80. Egress: 8*1024*0.09 = 737.28. "
                                "Total = 2796.66 + 1228.80 + 737.28 = 4762.74. "
                                "Wait — I'll recheck the discount. 3410.56 * 0.18 = 613.90, so after discount 2796.66. "
                                "Hmm, but also I should apply discount to the whole bill since reserved discount is global. "
                                "So 3410.56 + 1228.80 + 737.28 = 5376.64 * 0.82 = 4408.84.\n4408.84",  # applied discount to wrong scope
                                 360),
    "q6_compound_interest":   ("P = 250000. After Y1: 250000*1.042 = 260500. "
                                "After Y2: 260500*1.048 = 273004. "
                                "After Y3: 273004*1.051 = 286927.20. "
                                "After Y4: 286927.20*1.045 = 299838.92. "
                                "After Y5: 299838.92*1.053 = 315730.39.\n315730.39",  # correct!
                                 250),
}


@mlflow.trace(
    name="cot_solve",
    span_type=SpanType.LLM,
    attributes={"technique": "zero_shot_cot"},
)
def cot_solve(qid: str, question: str) -> tuple[str, Optional[float], int]:
    """Return (raw_text, parsed_answer, tokens_used)."""
    span = mlflow.get_current_active_span()
    prompt = COT_PROMPT.format(question=question)
    if span:
        span.set_inputs({"qid": qid, "question": question, "prompt": prompt,
                         "params": {"model": MODEL, "temperature": TEMPERATURE, "max_tokens": 600}})

    t0 = time.time()
    if USE_MOCK:
        text, tokens = _MOCK_COT[qid]
        latency_ms = (time.time() - t0) * 1000
        tag_cost_latency(latency_ms, 0.0)
        answer = parse_last_number(text)
        if span:
            span.set_outputs({"text": text, "answer": answer, "tokens": tokens})
        return text, answer, tokens

    resp = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=TEMPERATURE,
        max_tokens=600,
    )
    latency_ms = (time.time() - t0) * 1000
    cost = price_of(MODEL, resp.usage)
    tag_cost_latency(latency_ms, cost)
    text = resp.choices[0].message.content
    tokens = resp.usage.total_tokens if resp.usage else 0
    answer = parse_last_number(text)
    if span:
        span.set_outputs({"text": text, "answer": answer, "tokens": tokens})
    return text, answer, tokens


print("cot_solve ready.")

In [ ]:
# -- Run CoT on all 6 questions --------------------------------------------

TOLERANCE = 0.005  # 0.5% relative tolerance


def is_correct(predicted: Optional[float], gold: float, tol: float = TOLERANCE) -> bool:
    if predicted is None:
        return False
    if gold == 0:
        return abs(predicted) < tol
    return abs(predicted - gold) / abs(gold) <= tol


cot_rows = []
for q in eval_data:
    text, pred, tokens = cot_solve(q["id"], q["question"])
    correct = is_correct(pred, q["answer"])
    cot_rows.append({
        "id": q["id"],
        "gold": q["answer"],
        "cot_answer": pred,
        "cot_correct": correct,
        "cot_tokens": tokens,
    })
    flag = "OK " if correct else "FAIL"
    print(f"[{flag}] {q['id']:25s} gold={q['answer']!s:>14s}  cot={pred!s:>14s}")

cot_df = pd.DataFrame(cot_rows)
print(f"\nCoT accuracy: {cot_df['cot_correct'].mean():.1%}  ({cot_df['cot_correct'].sum()}/{len(cot_df)})")
print(f"CoT avg tokens: {cot_df['cot_tokens'].mean():.0f}")

## Watch one fail in slow motion

Before we move on, let's actually look at one of those CoT failures. Pick the worst offender — the one where the *reasoning* was algebraically correct but the *arithmetic* slipped a decimal — and print the full transcript. This is the moment that should make every engineer in the room wince.

Notice the structure of the failure:
1. Model identifies the right formula.
2. Model picks the right operands.
3. Model multiplies them in its head.
4. Model **second-guesses itself**, tries to recover, and commits to the wrong correction.

It's not a reasoning failure. It's a *computation* failure dressed up as reasoning.

In [ ]:
# -- Inspect one CoT failure in full -------------------------------------

failures = [r for r in cot_rows if not r["cot_correct"]]
if not failures:
    print("No CoT failures this run — try a higher temperature or re-run.")
else:
    # Pick the most dramatic failure (largest relative error)
    def rel_err(row):
        if row["cot_answer"] is None:
            return float("inf")
        return abs(row["cot_answer"] - row["gold"]) / max(abs(row["gold"]), 1e-9)

    worst = max(failures, key=rel_err)
    qid = worst["id"]
    # Get the original question and the full CoT response
    question = next(q["question"] for q in eval_data if q["id"] == qid)
    full_text, _, _ = cot_solve(qid, question)

    print(f"Question  : {qid}")
    print(f"Gold      : {worst['gold']}")
    print(f"CoT said  : {worst['cot_answer']}")
    print(f"Off by    : {rel_err(worst) * 100:.1f}% of gold")
    print("-" * 70)
    print("Full CoT transcript:")
    print("-" * 70)
    print(full_text)
    print("-" * 70)
    print("^ The reasoning structure is sound. The arithmetic is not.")

## PoT — let the interpreter compute

Instead of asking the model to *do* the arithmetic, we ask the model to **write Python that computes the answer**, and we run that Python in a sandboxed subprocess. The model handles the *symbolic reasoning* (which formula? which operands?). The interpreter handles the *numerical computation* (which the model is genuinely bad at).

Two pieces:
1. `safe_exec(code)` — `subprocess.run` with a 10-second timeout. No network, no filesystem access beyond a temp file. In production you'd use `restrictedpython`, a Docker container with `--network=none --read-only --cap-drop=ALL`, or a managed sandbox like E2B / Pyodide.
2. `pot_solve(question)` — prompt asks for a code block ending in `print(result)`. We extract the fenced code, exec it, parse the last line of stdout as a float.

**The key promise:** the model can now use `math.pow`, `numpy.percentile`, `pandas.groupby` — actual implementations of the operations it would otherwise approximate from memory.

In [ ]:
# -- safe_exec: sandboxed Python execution ---------------------------------
# Sandboxed: subprocess + 10s timeout, no in-process exec(), no network sandbox
# enforced here. For production add restrictedpython, Docker --network=none,
# seccomp filters, or a managed sandbox like E2B / Pyodide.

_ANSI = re.compile(r"\x1b\[[0-9;]*[A-Za-z]")


def safe_exec(code: str, timeout: int = 10) -> tuple[Optional[str], Optional[str]]:
    """Run Python code in a subprocess. Return (stdout, stderr_if_any).

    Sandboxed: subprocess + timeout, no network, restricted env.
    For production add restrictedpython or Docker --network=none.
    NEVER use in-process `exec()` — even on internal traffic, prompt-injection
    can land arbitrary Python in your output.
    """
    try:
        proc = subprocess.run(
            ["python3", "-c", code],   # subprocess: out-of-process; timeout; captured I/O.
            capture_output=True,
            text=True,
            timeout=timeout,
        )
        stdout = _ANSI.sub("", proc.stdout).strip()
        stderr = _ANSI.sub("", proc.stderr).strip()
        return (stdout, stderr if proc.returncode != 0 else None)
    except subprocess.TimeoutExpired:
        return (None, f"Timeout after {timeout}s")
    except Exception as e:
        return (None, f"{type(e).__name__}: {e}")


# Quick smoke test
out, err = safe_exec("print(2 + 2)")
assert out == "4", out
print("safe_exec OK  (subprocess + 10s timeout; no in-process exec).")

In [ ]:
# -- PoT: model writes Python, we exec it ----------------------------------

POT_PROMPT = (
    "Write a short Python script that computes the answer to the question below. "
    "Requirements:\n"
    "  - Use standard library or numpy/pandas only.\n"
    "  - Store the final numeric answer in a variable named `result`.\n"
    "  - End the script with exactly `print(result)`.\n"
    "  - Output ONLY a single fenced ```python code block — no prose, no explanation.\n\n"
    "Question: {question}"
)

_FENCE_RE = re.compile(r"```(?:python)?\s*\n?(.*?)```", re.DOTALL)


def extract_code(text: str) -> str:
    """Pull Python code out of a fenced block; fall back to the whole text."""
    m = _FENCE_RE.search(text)
    return m.group(1).strip() if m else text.strip()


# Mock PoT code blocks — pre-canned correct programs (interpreter does the arithmetic).
_MOCK_POT = {
    "q1_p95_latency": (
        "import numpy as np\n"
        "data = [42, 51, 38, 67, 49, 55, 71, 44, 58, 62, 39, 48, 53, 61, 47, 89, 52, 56, 41, 73]\n"
        "result = float(np.percentile(data, 95, method='linear'))\n"
        "result = round(result, 4)\n"
        "print(result)\n",
        180,
    ),
    "q2_quarterly_revenue": (
        "a, b, c = 120000, 85000, 45000\n"
        "for _ in range(3):\n"
        "    a *= 1.08\n"
        "    b *= 1.12\n"
        "    c *= 1.05\n"
        "result = round(a + b + c, 2)\n"
        "print(result)\n",
        160,
    ),
    "q3_token_cost": (
        "daily_cost = 1_200_000 * 2.50 / 1_000_000\n"
        "annual = daily_cost * 365\n"
        "result = round(annual * 1.15, 2)\n"
        "print(result)\n",
        150,
    ),
    "q4_fx_gross_margin": (
        "ty_rev_usd = 4_250_000 * 1.08\n"
        "ty_cogs_usd = 1_890_000 * 1.08\n"
        "ty_gm = (ty_rev_usd - ty_cogs_usd) / ty_rev_usd * 100\n"
        "ly_rev_usd = 3_800_000 * 1.12\n"
        "ly_cogs_usd = 1_710_000 * 1.12\n"
        "ly_gm = (ly_rev_usd - ly_cogs_usd) / ly_rev_usd * 100\n"
        "result = round(ty_gm - ly_gm, 4)\n"
        "print(result)\n",
        220,
    ),
    "q5_cloud_bill": (
        "compute = (12*730*0.096) + (6*730*0.192) + (4*730*0.340) + (8*730*0.126)\n"
        "compute_after = compute * (1 - 0.18)\n"
        "storage = 15 * 1024 * 0.08\n"
        "egress = 8 * 1024 * 0.09\n"
        "result = round(compute_after + storage + egress, 2)\n"
        "print(result)\n",
        210,
    ),
    "q6_compound_interest": (
        "p = 250_000\n"
        "for r in [0.042, 0.048, 0.051, 0.045, 0.053]:\n"
        "    p *= (1 + r)\n"
        "result = round(p, 2)\n"
        "print(result)\n",
        170,
    ),
}


@mlflow.trace(
    name="pot_solve",
    span_type=SpanType.CHAIN,
    attributes={"technique": "pot"},
)
def pot_solve(qid: str, question: str) -> tuple[str, Optional[float], int]:
    """Ask model for Python, exec in sandbox, parse stdout as float.

    Returns (code, parsed_answer, tokens_used).
    PoT code-generation call uses temperature=0.0 — reproducible programs.
    """
    span = mlflow.get_current_active_span()
    prompt = POT_PROMPT.format(question=question)
    if span:
        span.set_inputs({"qid": qid, "question": question, "prompt": prompt,
                         "params": {"model": MODEL, "temperature": 0.0, "max_tokens": 400}})

    t0 = time.time()
    if USE_MOCK:
        code, tokens = _MOCK_POT[qid]
        latency_ms = (time.time() - t0) * 1000
        tag_cost_latency(latency_ms, 0.0)
    else:
        resp = client.chat.completions.create(
            model=MODEL,
            messages=[{"role": "user", "content": prompt}],
            temperature=0.0,   # PoT executor MUST stay deterministic
            max_tokens=400,
        )
        latency_ms = (time.time() - t0) * 1000
        cost = price_of(MODEL, resp.usage)
        tag_cost_latency(latency_ms, cost)
        code = extract_code(resp.choices[0].message.content)
        tokens = resp.usage.total_tokens if resp.usage else 0

    # Subprocess exec gets its own child span so the tape recorder captures stdout/stderr.
    with mlflow.start_span(name="safe_exec", span_type=SpanType.TOOL) as exec_span:
        exec_span.set_inputs({"code": code})
        t1 = time.time()
        stdout, stderr = safe_exec(code)
        exec_latency_ms = (time.time() - t1) * 1000
        exec_span.set_attribute("latency_ms", exec_latency_ms)
        exec_span.set_attribute("cost_usd", 0.0)   # subprocess: no LLM cost
        exec_span.set_outputs({"stdout": stdout, "stderr": stderr})

    if stdout is None:
        if span:
            span.set_outputs({"code": code, "answer": None, "tokens": tokens,
                               "stdout": stdout, "stderr": stderr})
        return code, None, tokens
    # Parse last line of stdout as float
    last_line = stdout.strip().splitlines()[-1] if stdout.strip() else ""
    try:
        answer = float(last_line.replace(",", "").replace("$", "").strip())
    except ValueError:
        answer = parse_last_number(stdout)
    if span:
        span.set_outputs({"code": code, "answer": answer, "tokens": tokens,
                           "stdout": stdout, "stderr": stderr})
    return code, answer, tokens


print("pot_solve ready.")

In [ ]:
# -- Run PoT on all 6 questions --------------------------------------------

pot_rows = []
for q in eval_data:
    code, pred, tokens = pot_solve(q["id"], q["question"])
    correct = is_correct(pred, q["answer"])
    pot_rows.append({
        "id": q["id"],
        "pot_answer": pred,
        "pot_correct": correct,
        "pot_tokens": tokens,
    })
    flag = "OK " if correct else "FAIL"
    print(f"[{flag}] {q['id']:25s} gold={q['answer']!s:>14s}  pot={pred!s:>14s}")

pot_df = pd.DataFrame(pot_rows)
print(f"\nPoT accuracy: {pot_df['pot_correct'].mean():.1%}  ({pot_df['pot_correct'].sum()}/{len(pot_df)})")
print(f"PoT avg tokens: {pot_df['pot_tokens'].mean():.0f}")

In [ ]:
# -- The bake-off table ----------------------------------------------------

bake = cot_df.merge(pot_df, on="id")
bake = bake[["id", "gold",
             "cot_answer", "cot_correct", "cot_tokens",
             "pot_answer", "pot_correct", "pot_tokens"]]

print("=" * 90)
print("BAKE-OFF: Chain-of-Thought vs Program-of-Thought on financial reasoning")
print("=" * 90)
print(bake.to_string(index=False))
print("=" * 90)

cot_acc = bake["cot_correct"].mean()
pot_acc = bake["pot_correct"].mean()
delta_pp = (pot_acc - cot_acc) * 100

print(f"\n  CoT accuracy : {cot_acc:.1%}   avg tokens: {bake['cot_tokens'].mean():.0f}")
print(f"  PoT accuracy : {pot_acc:.1%}   avg tokens: {bake['pot_tokens'].mean():.0f}")
print(f"  Delta        : {delta_pp:+.1f} percentage points  (FinQA benchmark: +24.1 pp)")
print(f"  Token ratio  : {bake['pot_tokens'].mean() / max(bake['cot_tokens'].mean(), 1):.2f}x  (PoT is usually CHEAPER per question)")

# Tag the parent trace for cross-run analysis in MLflow
try:
    mlflow.update_current_trace(tags={
        "cot_accuracy": f"{cot_acc:.3f}",
        "pot_accuracy": f"{pot_acc:.3f}",
        "delta_pp":     f"{delta_pp:+.1f}",
        "n_questions":  str(len(bake)),
    })
except Exception as e:
    # update_current_trace requires an active trace context — fine to skip in notebook scope
    print(f"  (trace tag update skipped: {e})")

## The decimal-place story

Look at the bake-off table above. Two cases worth calling out:

**Q3 — the annual token bill.** CoT reasoned correctly: *"1.2M tokens at \$2.50/M, multiply by 365 days, multiply by 1.15."* Then it computed `1.2M × $2.50 / 1M = $3.00` — **off by 10×** (the right answer is \$0.30). The rest of the chain was algebraically correct on top of a wrong unit conversion, and it dutifully output \$125.93 instead of \$1259.25 (or vice versa). The model **noticed** the error mid-chain and tried to fix it, then committed to the wrong fix. **PoT** wrote `daily_cost = 1_200_000 * 2.50 / 1_000_000` and the interpreter computed `3.0` exactly — no decimal slip possible.

**Q5 — the cloud bill.** CoT enumerated every line item correctly. Then in the final summation it second-guessed itself on the *scope* of the 18% reserved-instance discount — applied it to compute+storage+egress instead of compute-only. Off by ~$350. **PoT** wrote:

```python
compute_after = compute * (1 - 0.18)
result = round(compute_after + storage + egress, 2)
```

The *scope* of the discount is encoded in the structure of the code, not in the model's working memory. Once it's typed, it can't drift.

**This is the FinQA benchmark in miniature.** Chen et al. measured PoT 64.5% vs CoT 40.4% on FinQA — a +24.1 percentage-point gap on tabular financial questions. The mechanism is exactly what you just saw: the LLM is good at picking the right operands and formulas (the *symbolic* part), and bad at multiplying and rounding them (the *numeric* part). PoT separates the two.

Open the MLflow UI at **http://127.0.0.1:5000** → experiment `session6/demo_05_pot_vs_cot` → look at the `cot_solve` and `pot_solve` traces side-by-side. The CoT spans show the prose reasoning chain in the LLM input/output. The PoT spans show the code generated, plus a child span for the subprocess execution.

## When PoT does NOT help

PoT is not a free win — it's a free win **for numeric tasks**. Cases where it actively hurts:

- **Non-numeric tasks.** Summarising a doc, classifying an intent, drafting an email — there's nothing for the interpreter to compute. PoT wraps prose in code and adds latency for no gain.
- **Tasks where the LLM doesn't know which library to call.** If your problem requires a domain-specific solver (e.g. mixed-integer programming, ODE solvers), the LLM may write code that looks plausible but uses the wrong algorithm. Test the generated code on a held-out eval before trusting it.
- **Tasks the LLM can answer from memory.** "What is 12 squared?" doesn't need a sandbox. The overhead of the subprocess (50–200ms) exceeds the answer-generation time.
- **Untrusted input.** If the question is constructed from user input, an attacker can prompt-inject code into the model's output that runs in your sandbox. Mitigations: `restrictedpython` for AST-level restrictions, Docker with `--network=none --read-only --cap-drop=ALL`, `seccomp` filters, or a managed sandbox like E2B / Pyodide. **Never** `exec(code)` in-process — always subprocess + timeout + resource limits, even on internal traffic.

Decision rule: if the task involves arithmetic with more than two operations on numbers larger than two digits, **default to PoT**. Otherwise, use CoT.

## Open the MLflow UI

1. MLflow server running? Visit **http://127.0.0.1:5000**.
2. Experiment: **`session6/demo_05_pot_vs_cot`** → Traces tab.
3. Side-by-side compare a `cot_solve` trace and a `pot_solve` trace:
   - `cot_solve` (LLM span): the input is the natural-language question; the output is the prose chain ending in a number. Click into the OpenAI sub-span to see token usage and the full message payload.
   - `pot_solve` (CHAIN span): the LLM sub-span shows the generated code; a sibling span captures the subprocess execution. The interpreter's `stdout` is the source of truth.
4. Filter traces by the tags written above (`cot_accuracy`, `pot_accuracy`, `delta_pp`) to compare across runs as you iterate on prompts.
5. **The "aha" moment to surface to learners:** the PoT trace contains a code block that any engineer can read, audit, and unit-test. The CoT trace contains 300 words of prose that nobody can audit at all. *That's* the production engineering argument, separate from the accuracy numbers.

## Takeaways

- **Don't let the LLM do mental arithmetic.** The reasoning is fine; the multiplication is not. Separate the two: LLM for *which formula*, interpreter for *which number*.
- **PoT is +8.5pp on GSM8K and +24.1pp on FinQA** (Chen et al. 2022, arXiv:2211.12588). The FinQA gap — tabular numeric reasoning — is the one that matters for engineering workloads.
- **Always sandbox the executor.** Subprocess + timeout + no network is the minimum bar. For untrusted input, escalate to Docker / `restrictedpython` / E2B. Cheap insurance against a prompt-injection RCE.
- **Cost overhead is tiny.** PoT prompts are *shorter* than CoT prompts (no "think step by step" preamble), and the model emits code instead of prose — often ~1.2× total tokens, sometimes less. The subprocess adds 50–200ms. For a +24pp accuracy gain on financial reasoning, this is the highest-ROI swap in the entire prompt-engineering toolkit.

**Reference:** Chen et al. 2022, *Program of Thoughts Prompting* — [arXiv:2211.12588](https://arxiv.org/abs/2211.12588). For the deeper symbolic generalisation, see Faithful CoT (Lyu et al. 2023, arXiv:2301.13379) covered in `03-wiki/06-secondary-techniques.md` section B10.

## ⚠️ Reasoning model caveat

This notebook assumes a **non-reasoning** base model. On `o1` / `o3` / Claude Extended Thinking / Gemini Thinking:
- Drop the CoT / ToT / Plan-and-Solve scaffolding — the model does it internally
- Use `developer` role instead of `system` on `o1-2024-12-17+`
- Never pass OpenAI `reasoning_effort` and Gemini `thinking_level` through the same OpenAI-compatible adapter
- See Block 5 of the slides for what's redundant vs what survives

PoT's argument **still applies on reasoning models** for the same reason it applies on non-reasoning ones: even with internal reasoning, the model is approximating arithmetic, not computing it. The interpreter is still the right tool for multi-step numeric work. Drop the CoT framing on the prompt, keep the PoT pattern.

## Replay every input + output in MLflow

Every prompt and every response from this demo is now in MLflow. Open the UI and click any trace to see the full I/O log.

In [ ]:
import IPython.display as D

ui = MLFLOW_TRACKING_URI.rstrip("/")
exp = mlflow.get_experiment_by_name(MLFLOW_EXPERIMENT)
exp_id = exp.experiment_id if exp else "0"
link = f"{ui}/#/experiments/{exp_id}?searchFilter=&orderByKey=attributes.start_time&orderByAsc=false"
D.display(D.Markdown(f"**Open the experiment in MLflow:** [{link}]({link})"))

recent = mlflow.search_traces(experiment_ids=[exp_id], max_results=5)
recent[["request_id", "timestamp_ms", "execution_time_ms", "tags"]] if not recent.empty else "No traces yet — run the cells above."